In [8]:
import pandas as pd
import ccxt
import ta


In [9]:
exchange = ccxt.coinsph () # default id
coinsph = ccxt.coinsph ({ 'id': 'okcoin1' })

In [11]:
import numpy as np

data_1d = exchange.fetch_ohlcv(symbol='BTC/PHP', timeframe='1d', since=None, limit=None, params={})
data_1w = exchange.fetch_ohlcv(symbol='BTC/PHP', timeframe='1w', since=None, limit=None, params={})
data_4h = exchange.fetch_ohlcv(symbol='BTC/PHP', timeframe='4h', since=None, limit=None, params={})
df = pd.DataFrame(data_1d)
#df.to_csv('Raw-btc.csv', index=False)
columns=['timestamp', 'open', 'high', 'low', 'close', 'volume']
cleaned_df1d = pd.DataFrame(data_1d, columns=columns)
cleaned_df1w = pd.DataFrame(data_1w, columns=columns)
cleaned_df4h = pd.DataFrame(data_4h, columns=columns)
cleaned_df1d["timestamp"] = pd.to_datetime(cleaned_df1d["timestamp"], unit="ms")
cleaned_df1w["timestamp"] = pd.to_datetime(cleaned_df1w["timestamp"], unit="ms")
cleaned_df4h["timestamp"] = pd.to_datetime(cleaned_df4h["timestamp"], unit="ms")
#cleaned_df1d.to_csv('Cleaned-btc.csv', index=False)
#cleaned_df1w.to_csv('Cleaned-btc-weekly.csv', index=False)
#cleaned_df4h.to_csv('Cleaned-btc-4h.csv', index=False)

In [ ]:

from fileinput import close


df = pd.read_csv('Cleaned-btc.csv')
df['EMA20'] = ta.trend.ema_indicator(df['close'], window=20)
df['EMA50'] = ta.trend.ema_indicator(df['close'], window=50)
df['EMA200'] = ta.trend.ema_indicator(df['close'], window=200)
df['mavolume'] = df['volume'].rolling(window=20).mean()
df['Prev_EMA_20'] = df['EMA20'].shift(1)
df['Prev_EMA_50'] = df['EMA50'].shift(1)
df1 = df[['high', 'low', 'close', 'EMA20', 'Prev_EMA_20']].tail(50)
df['RSI'] = ta.momentum.rsi(df['close'], window=14)
df['EMA_200'] = ta.trend.ema_indicator(df['close'], window=200)

df['ATR'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=14)

df['Trend_Bullish'] = df['close'] > df['EMA_200']
df['RSI_Recently_Oversold'] = (df['RSI'] < 35).rolling(window=14).max() == 1 
df['RSI_Overbought'] = df['RSI'] > 65

ram_db = {}
df_clean = df.dropna()

def buy(data):
    threshold = 1.3

    has_increased = data['volume'].pct_change() > threshold #volume confirmation
    macro_filter = (data['EMA50'] >= data['EMA200']) & (data['EMA50'].shift(1) < data['EMA200'].shift(1)) #fast EMA cross
    Momentum_filter = (data['EMA20'] >= data['EMA50']) & (data['EMA20'].shift(1) < data['EMA50'].shift(1)) #fast EMA cross
    trend = data['Trend_Bullish'] #RSI MULTI FILTER
    rsi_oversold = data['RSI_Recently_Oversold'] #RSI MULTI FILTER
    waiting = (data['EMA20'] > data['EMA50']) & (data['close'].rolling(window=10).max() > data['EMA20'] * 1.03) & ((data['EMA50'] < data['close']) & (data['close'] < data['EMA20'])) #waiting for price to come back to EMA20
    stop_loss = data['close'] - (data['ATR'] * 2) #stop loss calculation
    trade_object = {
        "Ticker": "btc",
        "Entry": 100,
        "Stop_Loss": 96,
        "Status": "Active"
    }
    return stop_loss, (has_increased and macro_filter and trend and rsi_oversold and Momentum_filter and waiting)

def memory(data, stored_num):
    ticker_name = stored_num["Ticker"]
    ram_db[ticker_name] = stored_num

def sell(data, math):
    ram_db['btc'] = stop_loss
    


    return


df_clean['Buy_Signal'] = buy(df_clean)
df_clean['Sell_Signal'] = sell(df_clean)
